# Real Estate Market Analytics – Exploratory Data Analysis

This notebook answers the 20 EDA questions from the project brief. It uses the cleaned dataset generated by `data_cleaning.py`.

**Scope:** Data cleaning and EDA only; no machine learning or MLflow.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data" / "cleaned_data.csv").exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / "data" / "cleaned_data.csv")
sns.set_theme(style="whitegrid")
print("Shape:", df.shape)
display(df.head())
display(df.describe(include="all").T)

## 1–5. Price and size analysis

### 1. What is the distribution of property prices?

In [ ]:
plt.figure(figsize=(9,4))
sns.histplot(df["Price_in_Lakhs"], bins=40, kde=True)
plt.title("Distribution of property prices"); plt.xlabel("Price (₹ Lakhs)"); plt.show()

### 2. What is the distribution of property sizes?

In [ ]:
plt.figure(figsize=(9,4))
sns.histplot(df["Size_in_SqFt"], bins=40, kde=True)
plt.title("Distribution of property sizes"); plt.xlabel("Size (sq. ft.)"); plt.show()

### 3. How does price per sq ft vary by property type?

In [ ]:
display(df.groupby("Property_Type")["Price_per_SqFt"].mean().sort_values(ascending=False).to_frame("Average ₹ per sq. ft."))
sns.barplot(data=df, x="Property_Type", y="Price_per_SqFt", estimator="mean", errorbar=None)
plt.title("Average price per sq. ft. by property type"); plt.ylabel("₹ per sq. ft."); plt.xticks(rotation=20); plt.show()

### 4. Is there a relationship between property size and price?

In [ ]:
sample = df.sample(min(10000, len(df)), random_state=42)
sns.scatterplot(data=sample, x="Size_in_SqFt", y="Price_in_Lakhs", alpha=.35)
plt.title("Property size vs price"); plt.xlabel("Size (sq. ft.)"); plt.ylabel("Price (₹ Lakhs)"); plt.show()
print("Pearson correlation:", df["Size_in_SqFt"].corr(df["Price_in_Lakhs"]))

### 5. Are there outliers in price per sq ft or property size?

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
sns.boxplot(y=df["Price_per_SqFt"], ax=axes[0]); axes[0].set_title("Price per sq. ft. outliers")
sns.boxplot(y=df["Size_in_SqFt"], ax=axes[1]); axes[1].set_title("Property size outliers")
plt.tight_layout(); plt.show()
print("Price/sq.ft. outlier flags:", int(df["Price_per_SqFt_Outlier_Flag"].sum()))
print("Size outlier flags:", int(df["Size_Outlier_Flag"].sum()))

## 6–10. Location-based analysis

### 6. What is the average price per sq ft by state?

In [ ]:
state = df.groupby("State")["Price_per_SqFt"].mean().sort_values(ascending=False)
display(state.to_frame("Average ₹ per sq. ft."))
state.plot(kind="bar", figsize=(12,4), title="Average price per sq. ft. by state"); plt.ylabel("₹ per sq. ft."); plt.show()

### 7. What is the average property price by city?

In [ ]:
city = df.groupby("City")["Price_in_Lakhs"].mean().sort_values(ascending=False)
display(city.head(15).to_frame("Average price (₹ Lakhs)"))
city.head(15).plot(kind="bar", figsize=(12,4), title="Top 15 cities by average property price"); plt.ylabel("₹ Lakhs"); plt.show()

### 8. What is the median age of properties by locality?

In [ ]:
locality_age = df.groupby("Locality")["Age_of_Property"].median().sort_values(ascending=False)
display(locality_age.head(20).to_frame("Median age (years)"))

### 9. How is BHK distributed across cities?

In [ ]:
top_cities = df["City"].value_counts().head(12).index
ct = pd.crosstab(df.loc[df["City"].isin(top_cities), "City"], df.loc[df["City"].isin(top_cities), "BHK"])
display(ct)
ct.plot(kind="bar", stacked=True, figsize=(12,5), title="BHK distribution across top 12 cities"); plt.ylabel("Property count"); plt.xticks(rotation=35); plt.show()

### 10. What are the price trends for the top 5 most expensive localities?

In [ ]:
top5 = df.groupby("Locality")["Price_in_Lakhs"].mean().nlargest(5)
display(top5.to_frame("Average listed price (₹ Lakhs)"))
top5.plot(kind="bar", figsize=(8,4), title="Top 5 localities by average listed price"); plt.ylabel("₹ Lakhs"); plt.show()

## 11–15. Feature relationships and correlation

### 11. How are numeric features correlated with each other?

In [ ]:
corr = df.select_dtypes("number").drop(columns=["ID"], errors="ignore").corr()
plt.figure(figsize=(12,8)); sns.heatmap(corr, cmap="coolwarm", center=0); plt.title("Numeric feature correlation"); plt.show()

### 12. How do nearby schools relate to price per sq ft?

In [ ]:
display(df.groupby("Nearby_Schools")["Price_per_SqFt"].mean().to_frame("Average ₹ per sq. ft."))
sns.lineplot(data=df, x="Nearby_Schools", y="Price_per_SqFt", errorbar=None, marker="o")
plt.title("Nearby schools vs price per sq. ft."); plt.show()

### 13. How do nearby hospitals relate to price per sq ft?

In [ ]:
display(df.groupby("Nearby_Hospitals")["Price_per_SqFt"].mean().to_frame("Average ₹ per sq. ft."))
sns.lineplot(data=df, x="Nearby_Hospitals", y="Price_per_SqFt", errorbar=None, marker="o")
plt.title("Nearby hospitals vs price per sq. ft."); plt.show()

### 14. How does price vary by furnished status?

In [ ]:
display(df.groupby("Furnished_Status")["Price_in_Lakhs"].mean().to_frame("Average price (₹ Lakhs)"))
sns.barplot(data=df, x="Furnished_Status", y="Price_in_Lakhs", errorbar=None)
plt.title("Average property price by furnishing"); plt.ylabel("₹ Lakhs"); plt.show()

### 15. How does price per sq ft vary by property facing direction?

In [ ]:
display(df.groupby("Facing")["Price_per_SqFt"].mean().to_frame("Average ₹ per sq. ft."))
sns.barplot(data=df, x="Facing", y="Price_per_SqFt", errorbar=None)
plt.title("Price per sq. ft. by facing direction"); plt.show()

## 16–20. Market, amenities and ownership analysis

### 16. How many properties belong to each owner type?

In [ ]:
display(df["Owner_Type"].value_counts().to_frame("Property count"))
sns.countplot(data=df, x="Owner_Type"); plt.title("Properties by owner type"); plt.show()

### 17. How many properties are available under each availability status?

In [ ]:
display(df["Availability_Status"].value_counts().to_frame("Property count"))
sns.countplot(data=df, x="Availability_Status"); plt.title("Properties by availability status"); plt.xticks(rotation=15); plt.show()

### 18. Does parking space affect property price?

In [ ]:
display(df.groupby("Parking_Space")["Price_in_Lakhs"].agg(["count","mean","median"]))
sns.boxplot(data=df, x="Parking_Space", y="Price_in_Lakhs", showfliers=False)
plt.title("Property price by parking availability"); plt.ylabel("₹ Lakhs"); plt.show()

### 19. How do amenities affect price per sq ft?

In [ ]:
display(df.groupby("Amenities_Count")["Price_per_SqFt"].mean().to_frame("Average ₹ per sq. ft."))
sns.barplot(data=df, x="Amenities_Count", y="Price_per_SqFt", errorbar=None)
plt.title("Amenities count vs price per sq. ft."); plt.show()

### 20. How does public transport accessibility relate to price per sq ft or investment potential?

In [ ]:
display(df.groupby("Public_Transport_Accessibility")["Price_per_SqFt"].agg(["count","mean","median"]))
sns.barplot(data=df, x="Public_Transport_Accessibility", y="Price_per_SqFt",
            order=["Low","Medium","High"], errorbar=None)
plt.title("Transport accessibility vs price per sq. ft."); plt.ylabel("₹ per sq. ft."); plt.show()
print("This is a descriptive comparison, not a measured investment-return estimate.")